# Phase2: Factors Construction

In [1]:
pip install torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from pathlib import Path
from math import sqrt
import os
import torch
import torch.nn as nn

In [4]:
from operators_library import ts_mean, ts_av_diff, ts_std_dev, ts_sum, ts_zscore, zscore, rank, ts_rank, signed_power, bucket, densify, ts_delay, group_neutralize, scale, trade_when, trade_when_hold, hump, vector_neut, group_mean, quantile_cauchy, ts_quantile_cauchy, ts_regression, ts_step, group_rank, winsorize, ts_backfill, ts_arg_min, ts_decay_linear, if_else, group_scale, days_from_last_change, mlp1d_apply, fit_mlp1d_sharpe, safe_div, ts_min,group_winsorize_std,grouped_panel

# **Options**

# Factor1

In [5]:
def alpha_0620b(
    historical_volatility_120,
    implied_volatility_call_90,
    implied_volatility_put_90,
    pcr_oi_270,
    pcr_oi_180,
    pcr_oi_10,
    returns,
):
  group = densify(bucket(rank(ts_mean(historical_volatility_120,10)),range='0.1,1,0.2'))
  alpha = ts_mean(ts_delay(implied_volatility_call_90 - implied_volatility_put_90,1),20)
  group_alpha = group_neutralize(alpha * ts_mean(ts_std_dev(alpha,1500),30),group)
  entry_condition = ts_mean(pcr_oi_270,5) < 1 & ts_mean(pcr_oi_180,5) < 1
  exit_condition = ts_mean(pcr_oi_10,5) > 7
  a = trade_when(entry_condition,scale(group_alpha),exit_condition)
  aa = hump(a,hump=0.0001)
  b = abs(ts_mean(returns,252)/ts_std_dev(returns,252))
  alpha_final = vector_neut(aa,b)
  return alpha_final

# Factor2

In [5]:
def alpha_0427(
    historical_volatility_120,
    implied_volatility_call_90,
    implied_volatility_put_90,
    pcr_oi_270,
):
  group = bucket(rank(ts_mean(historical_volatility_120,2)),range='0.1,1,0.1')
  alpha = ts_mean(implied_volatility_call_90 - implied_volatility_put_90,15)
  group_alpha = group_neutralize(alpha,group)
  alpha_final = trade_when(ts_mean(pcr_oi_270,5)<1,scale(group_alpha),-1)
  return alpha_final

# **Price-Volume**

# Factor1

In [6]:
def alpha_0617a(
    open,
    volume
):
    A = ts_regression(open,open,window=5,lag=1,rettype=3)
    B = ts_mean(ts_delay(open,1),window=2)
    C = (A-B)/B
    D = 1 - rank(volume/ts_mean(volume,500))
    aa = -ts_rank(C,126)*D
    # if a stock is in top20% of |C| today, double its signal
    absC_rank = rank(C.abs())
    boost_mask = absC_rank > 0.98
    aaa = aa.where(~boost_mask , aa*3.0)
    alpha_final = hump(aaa,hump=0.0185)
    return alpha_final

# Factor2

In [7]:
def alpha_0608d(
    close,
    open,
    volume,
    sharesout,
    industry,
    horro_window
):
    intra_ret = close/open - 1
    mean_returns = group_mean(intra_ret,rank(ts_mean(volume*sharesout,10)),industry)
    horro = abs(intra_ret-mean_returns)/(abs(intra_ret)+abs(mean_returns)+0.1)
    horro_day = ts_mean(horro,horro_window)
    ret_std = ts_std_dev(intra_ret,horro_window)
    adj_ret = horro_day*ret_std*intra_ret
    adj_ret_mean = ts_mean(adj_ret,horro_window) 
    adj_ret_std = ts_std_dev(adj_ret,horro_window)
    horro_std_bonus = zscore(adj_ret_mean) + zscore(adj_ret_std)
    a = -quantile_cauchy(horro_std_bonus,eps=1e-6)
    cond1 = ts_rank(ts_mean(volume,3)/ts_mean(volume,280),252) > 0.52
    cond2 = ts_mean(volume,3)/ts_mean(volume,252) > 1.01
    aa = a.where(cond1|cond2,-1.0)
    alpha_final = hump(aa,0.028)
    return alpha_final

# Factor3

In [8]:
def alpha_0810a(
    returns,
    cap,
    volume,
): 
    a = ts_regression(returns,ts_step(returns),window=65,lag=0,rettype=0)
    group = densify(bucket(rank(cap), 0.0,1.0,0.2))
    aa = -group_rank(ts_sum(a,5),group)
    Rsquare = ts_regression(returns,ts_step(returns),window=65,lag=0,rettype=6)
    
    vol_short = ts_mean(volume, 2)
    vol_long  = ts_mean(volume, 252)
    # liquidity condition
    condition_liq = (vol_short / vol_long) > 2.0

    rsq_rank = rank(Rsquare)                 
    condition_r2 = rsq_rank > 0.45
    condition_liq = condition_liq.reindex_like(condition_r2)
    
    cond_values = np.logical_and(condition_liq.values, condition_r2.values)
    cond = pd.DataFrame(cond_values,
                    index=condition_r2.index,
                    columns=condition_r2.columns)
    aaa = trade_when_hold(cond,aa)
    alpha_final = winsorize(aaa,std=0.6)
    return alpha_final

# Factor4

In [9]:
def alpha_0415b(
    close,
    volume
):
    A = ts_regression(close,close,window=5,lag=1,rettype=3)
    B = ts_mean(ts_delay(close,1),window=1)
    C = (A-B)/B
    D = 1 - rank(volume/ts_mean(volume,504))
    aa = -ts_rank(C,120)*D
    alpha_final = hump(aa,hump=0.00003)
    return alpha_final

# Factor5

In [10]:
def alpha_0616a(
    close,
    open,
    volume
):
    A = ts_regression(close,open,window=20,lag=1,rettype=3)
    B = ts_mean(ts_delay(open,2),window=2)
    C = (A-B)/B
    D = 1 - rank(volume/ts_mean(volume,504))
    aa = -ts_rank(C,15)*D
    alpha_final = hump(aa,hump=0.0015)
    return alpha_final

# Factor6

In [11]:
def alpha_0604c(
    close,
    high,
    low
):
    STO = (close-low)/(high-low)
    STO_smoothed = ts_mean(STO,6)
    signal = if_else(STO_smoothed<0.3,1,if_else(STO_smoothed>0.9,-1,0))
    alpha_final = ts_decay_linear(ts_rank(signal,110)**0.2,45)
    return alpha_final

# Factor7

In [12]:
def alpha_0413a(
    close,
    high,
    low
):
    alpha_final = rank(ts_sum((close-high)/(low-close),1))
    return alpha_final

# Factor8

In [13]:
def alpha_0503b(
    sharesout,
    volume,
    vwap,
    low,
    open,
    high
):
    term1 = -18 * sharesout/volume
    term2 = 0.3 * rank(vwap**1/3)/rank(low**1/3)
    term3 = 0.1* ts_regression(open,high,21,lag=0,rettype=2)
    alpha_final = term1 - term2 + term3
    return alpha_final

# Factor9

In [14]:
def alpha_0603a(
        volume,
        sharesout,
        cap
):
    turnover = volume/sharesout
    signal = group_rank(ts_av_diff(turnover,1000),bucket(rank(cap),0.1,1,0.1))
    smoothed_signal = ts_mean(signal,2)
    alpha_final = trade_when_hold(abs(ts_zscore(turnover,252))<0.6,smoothed_signal ** 2.5)
    return alpha_final

# Factor10

In [15]:
def alpha_0416a(
        returns,
        subindustry,
        volume
):
    alpha = hump(group_scale(-returns,subindustry)**0.5,hump=0.0002)
    alpha_final = trade_when(ts_decay_linear(volume,3)/ts_mean(volume,125)>1,alpha,ts_decay_linear(volume,3)/ts_mean(volume,125)<0.1)
    return alpha_final

# Factor11

In [ ]:
def alpha_0421b(
        close,
        open,
        volume,
        returns
):
    RET_OCplus1 = close/open
    RET_OC = RET_OCplus1 -1
    RETplus1 = close / (ts_delay(close,1))
    RET_CO = RETplus1/RET_OCplus1 - 1
    cond = ((RET_CO > 0) & (RET_OC < 0)).fillna(False)
    NUM = if_else(cond, 1,0)
    NR = ts_mean(NUM,20)
    AB_NR = NR/ts_mean(NR,250)
    alpha = trade_when_hold(ts_mean(volume,5)/ts_mean(volume,250)>1,rank(AB_NR)+rank(-returns))
    alpha_final = hump(alpha,hump=0.001)
    return alpha_final

# Factor12

In [17]:
def alpha_0519d(
        split,
        volume
):
    a = days_from_last_change(split)
    aa = trade_when_hold(ts_mean(volume,5)/ts_mean(volume,252)>1,a**0.02)
    alpha_final = hump(aa,hump=0.00005)
    return alpha_final

# Factor13

In [18]:
def alpha_0403c(
        volume,
        sharesout
):
    turnover = volume/sharesout
    signal = rank(ts_av_diff(turnover,252))
    smoothed_signal = ts_mean(signal,10)
    alpha_final = trade_when_hold(abs(ts_zscore(turnover,252))<0.7,smoothed_signal)
    return alpha_final

# Factor14

In [19]:
def alpha_0618e(
        vwap,
        close,
        volume
):
    a = signed_power((vwap-close)/close,0.9)*volume/ts_decay_linear(volume,5)
    mask = ts_rank(ts_std_dev(volume,252),252)>0.96
    alpha_final = a.where(~mask,a/2)
    return alpha_final

# **Fundamentals**

# Factor1

In [20]:
def alpha_0812a(
    liabilities,
    assets,
    debt,
    equity
):
    a = ts_backfill(liabilities/assets,120)
    b = ts_arg_min(debt/equity,504) ** 2
    alpha_final = winsorize(a*b,std=0.3)
    return alpha_final

# Factor2

In [ ]:
def alpha_0611(
    enterprise_value,
    ebitda,
    volume,
    subindustry,
):
    signal = group_rank(-ts_zscore(enterprise_value/ebitda,63)/ts_mean(volume,15),subindustry)
    base_signal = signal ** 0.3
    vol_11 = ts_mean(volume,11)
    vol_252 = ts_mean(volume,252)
    cond_liq = (vol_11/vol_252) > 1.12
    a = trade_when_hold(cond_liq,base_signal)
    alpha_final = hump(a,hump=0.0002)
    return alpha_final

# Factor3

In [22]:
def alpha_0612d(
    debt,
    assets,
    volume
):
    a = -ts_quantile_cauchy(debt/assets,504)*ts_std_dev(debt,63)
    vol_40 = ts_mean(volume,40)
    vol_252 = ts_mean(volume,252)
    cond_liq = vol_40/vol_252 > 1.0
    alpha_final = trade_when_hold(cond_liq,a)
    return alpha_final

# Factor4

In [23]:
def alpha_0615a(
    volume,
    sharesout,
    cap
):
    turnover = volume/sharesout
    signal = group_rank(ts_av_diff(turnover,1008),bucket(rank(cap),0.0,1.0,0.2))
    smoothed_signal1 = ts_mean(signal,3)
    turnover_z = ts_zscore(turnover,252)
    cond = (turnover_z.abs()<0.3)
    a1 = trade_when_hold(cond,smoothed_signal1)
    b1 = hump(a1,hump=0.0003)

    smoothed_signal2 = ts_mean(signal,2)
    a2 = trade_when_hold(cond,smoothed_signal2)
    b2 = hump(a2,hump=0.0001)

    alpha_final = b1 + b2
    return alpha_final

# Factor5

In [24]:
def alpha_0407a(
    debt
):
    alpha_final = -ts_quantile_cauchy(debt,window=22)
    return alpha_final

# Factor6

In [25]:
def alpha_0505(
    operating_income,
    vwap
):
    ratio = operating_income/vwap
    ranking = ts_rank(ratio,252)
    score = ranking ** 2
    cond = score > 0.9
    alpha_final = if_else(cond,2,-1)
    alpha_final = pd.DataFrame(alpha_final, index=ratio.index, columns=ratio.columns)
    return alpha_final

# Factor7

In [26]:
def alpha_0413b(
    operating_income,
    cap
):
    alpha_final = ts_rank(operating_income/cap,42)
    return alpha_final

# Factor8

In [27]:
def alpha_0404b(
        returns,
        operating_income,
        cap
):
    momentum = ts_mean(returns,5)
    OEY = operating_income/cap
    smooth_OEY = ts_mean(OEY,3)
    OEY_rank = ts_rank(smooth_OEY,125)
    alpha_final = trade_when_hold(momentum<0.1,OEY_rank)
    return alpha_final

# Factor9

In [28]:
def alpha_0412(
        operating_income,
        vwap,
        volume,
        returns
):
    alpha = ts_rank(operating_income/(vwap*ts_mean(volume,60)),252)
    momentum = ts_mean(returns,5)
    alpha_final = if_else(momentum<0.8,alpha,ts_mean(alpha,10))
    return alpha_final

# Factor10

In [29]:
def alpha_0421a(
        enterprise_value,
        ebitda,
        subindustry
):
    alpha = group_rank(-ts_zscore(enterprise_value/ebitda,250),subindustry)
    alpha_final = alpha ** 0.5
    return alpha_final

# Factor11

In [35]:
def alpha_0418b(
        operating_income,
        vwap
):
    a = ts_rank(operating_income/vwap,126)
    alpha_final = a**4
    return alpha_final

# Factor12

In [31]:
def alpha_0604a(
    operating_income,
    assets,
    volume,
    cap
):
    OEY = ts_backfill(operating_income, 2) / ts_backfill(assets, 2)
    smooth_OEY = ts_mean(OEY, 5)
    OEY_rank = ts_rank(smooth_OEY, 185)
    
    a = trade_when_hold(
        ts_mean(volume, 5) / ts_mean(volume, 300) > 1.05, OEY_rank ** 0.2
    )
    mask = rank(days_from_last_change(ts_backfill(operating_income,2)))<0.2
    alpha_final = a.where(~mask,a*1.2)
    return alpha_final


# Factor13

In [5]:
def alpha_pe_value_profit_gate(
    pe,             
    net_profit,        
    industry,            
    profit_window: int = 252*3,       # parameter: "3 yrs" by default
    pe_clip_std: float = 2.5,         # group winsorize strength
    smooth_window: int = 10,          # optional smoothing
):
    pe, net_profit = pe.align(net_profit, join="inner", axis=0)
    pe, net_profit = pe.align(net_profit, join="inner", axis=1)

    # --- make industry/subindustry panel once
    group_panel = grouped_panel(pe,industry)

    # --- (1) group-wise outlier clip on PE
    pe_clip = group_winsorize_std(pe, industry, n_std=pe_clip_std)

    # --- (2) value signal = industry-relative rank: low PE should be long
    # group_rank returns pct rank within group (ascending): low PE -> small rank
    r = group_rank(pe_clip, industry)
    value = (0.5 - r)               

    # --- (3) profitability gate: net_profit > 0 for the whole rolling window
    profit_ok = (ts_min(net_profit, profit_window) > 0)
    sig = if_else(profit_ok, value, 0.0)

    if smooth_window and smooth_window > 1:
        sig = sig.rolling(smooth_window, min_periods=1).mean()

    # --- (5) optional final cross-sectional clip (global, not group)
    # winsorize() exists :contentReference[oaicite:7]{index=7}
    sig = winsorize(sig, std=3.0)
    return sig

# Extra Factors

## MLP Non Linear Momentum

In [32]:
# simple cache so we don't retrain every time you call the alpha
_NLTSMOM_CACHE = {}

def alpha_nltsmom_ann(
    close,                   # DataFrame: (date x stock)
    horizon=63,
    vol_window=252,
    train_frac=0.6,
    val_frac=0.2,
    n_hidden=16,
    epochs=40,
    lr=1e-3,
    weight_decay=1e-3,
    batch_size=200000,
    max_samples=2_000_000,
    seed=42,
    clip_s=8.0,
    winsor=3.0,
    do_cs_rank=True,
):
    """
    NLTSMOM (Empirical ANN mapping):
      1) s_t = sqrt(h) * mean(r_{t-h:t-1}) / sigma_{t|t-1}
      2) y_t = r_t / sigma_{t|t-1}
      3) fit f(s) to maximize Sharpe of f(s)*y on train, choose best via val
      4) output factor = scaled f(s) (optionally winsorized + cs-ranked)
    """
    returns = close.pct_change()
    # --- 1) build ex-ante components (use info up to t-1)
    r_lag = ts_delay(returns, 1)
    mu = ts_mean(r_lag, horizon)
    sig = ts_std_dev(r_lag, vol_window)  # daily sigma estimate, ex-ante

    s = np.sqrt(horizon) * safe_div(mu, sig)
    s = s.clip(-clip_s, clip_s)

    y = safe_div(returns, sig)  # risk-adjusted realized return at t

    # --- 2) fit mapping once (cached)
    key = (horizon, vol_window, train_frac, val_frac, n_hidden, epochs, lr, weight_decay, batch_size, max_samples, seed)
    if key not in _NLTSMOM_CACHE:
        params = fit_mlp1d_sharpe(
            s, y,
            train_frac=train_frac,
            val_frac=val_frac,
            n_hidden=n_hidden,
            epochs=epochs,
            batch_size=batch_size,
            lr=lr,
            weight_decay=weight_decay,
            seed=seed,
            max_samples=max_samples,
            scale_mode="unit_var",
        )
        _NLTSMOM_CACHE[key] = params

    params = _NLTSMOM_CACHE[key]

    # --- 3) apply mapping
    f = mlp1d_apply(s, params["w1"], params["w2"]) * params["scale"]

    # --- 4) optional post-processing consistent with your pool style
    f = winsorize(f,std=winsor)
    if do_cs_rank:
        f = rank(f)

    return f